In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### 一、导入数据并概览

In [ ]:
from pathlib import Path

DATA_DIR = Path('../data')

df = pd.read_csv(DATA_DIR / 'train.csv')
df.head()

##### 1.字段结构概览

In [ ]:
print(df.info())
print(f'\nColumns: \n{df.columns}')

In [ ]:
col = ['channelGrouping', 'socialEngagementType']
for i in col:
    print(f'{i}:')
    print(sorted(df[i].unique()))

##### 2.JSON样本概览

In [ ]:
col = ['device', 'geoNetwork', 'totals', 'trafficSource']
for i in col:
    print(f'{i}:\n{df.iloc[102][i]}')

##### 3.处理时间字段

In [ ]:
df['visitStartTime'] = pd.to_datetime(df['visitStartTime'], unit='s', utc=True).dt.tz_convert('America/Los_Angeles').dt.tz_localize(None)
print(df['visitStartTime'].min())
print(df['visitStartTime'].max())

### 二、数据清洗

##### 1.删除无用列

In [ ]:
drop_col = ['date', 'socialEngagementType', 'sessionId', 'visitId']
df.drop(columns=drop_col, inplace=True)

##### 2.展开JSON列

In [ ]:
import json
from pandas import json_normalize

# 自定义展开函数
def flatten_json_columns(df, json_cols):
    df = df.copy()
    for col in json_cols:
        # 字符串转换字典
        print(f'正在处理:{col}')
        df[col] = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

        # 展开json并定义列名为来源+小标题
        flat_df = json_normalize(df[col])
        flat_df.columns = [f"{col}.{subcol}" for subcol in flat_df.columns]

        # 重置索引
        df = df.reset_index(drop=True)
        flat_df = flat_df.reset_index(drop=True)

        # 合并删除原列
        df = pd.concat([df.drop(col, axis=1), flat_df], axis=1)

    return df

In [ ]:
json_cols = ['device', 'geoNetwork', 'totals', 'trafficSource']
df = flatten_json_columns(df, json_cols)
df.head()

##### 3.删除展开的JSON中的无用列

In [ ]:
print(df.columns.to_list())

In [ ]:
cols = ['device.browser', 'device.browserVersion', 'device.browserSize', 'device.operatingSystem', 'device.operatingSystemVersion', 'device.isMobile'
        , 'device.mobileDeviceBranding', 'device.mobileDeviceModel', 'device.mobileInputSelector', 'device.mobileDeviceInfo'
        , 'device.mobileDeviceMarketingName', 'device.flashVersion', 'device.language', 'device.screenColors', 'device.screenResolution'
        , 'device.deviceCategory', 'geoNetwork.continent', 'geoNetwork.subContinent', 'geoNetwork.country', 'geoNetwork.region', 'geoNetwork.metro'
        , 'geoNetwork.city', 'geoNetwork.cityId', 'geoNetwork.networkDomain', 'geoNetwork.latitude', 'geoNetwork.longitude'
        , 'geoNetwork.networkLocation', 'totals.visits', 'totals.hits', 'totals.pageviews', 'totals.bounces', 'totals.newVisits'
        , 'totals.transactionRevenue', 'trafficSource.campaign', 'trafficSource.source', 'trafficSource.medium', 'trafficSource.keyword'
        , 'trafficSource.adwordsClickInfo.criteriaParameters', 'trafficSource.isTrueDirect', 'trafficSource.referralPath'
        , 'trafficSource.adwordsClickInfo.page', 'trafficSource.adwordsClickInfo.slot', 'trafficSource.adwordsClickInfo.gclId'
        , 'trafficSource.adwordsClickInfo.adNetworkType', 'trafficSource.adwordsClickInfo.isVideoAd', 'trafficSource.adContent'
        , 'trafficSource.campaignCode']
for i, col in enumerate(cols):
    print(f'\n{i+1}.{col}:\n{df[col].unique()}')

In [ ]:
col = ['device.browserVersion', 'device.browserSize', 'device.operatingSystemVersion', 'device.mobileDeviceBranding', 'device.mobileDeviceModel'
       , 'device.mobileInputSelector', 'device.mobileDeviceInfo', 'device.mobileDeviceMarketingName', 'device.flashVersion', 'device.language'
       , 'device.screenColors', 'device.screenResolution'
       , 'geoNetwork.subContinent', 'geoNetwork.metro', 'geoNetwork.cityId', 'geoNetwork.networkDomain', 'geoNetwork.latitude', 'geoNetwork.longitude'
       , 'geoNetwork.networkLocation', 'totals.visits'
       , 'trafficSource.keyword', 'trafficSource.adwordsClickInfo.criteriaParameters', 'trafficSource.referralPath'
       , 'trafficSource.adwordsClickInfo.page', 'trafficSource.adwordsClickInfo.gclId', 'trafficSource.adwordsClickInfo.isVideoAd'
       , 'trafficSource.adContent', 'trafficSource.campaignCode']
df.drop(columns=col, inplace=True)
df.head()

##### 4.处理JSON数值

In [ ]:
#填充缺失值
cols = ['totals.hits', 'totals.pageviews', 'totals.bounces', 'totals.newVisits']
for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
df['totals.transactionRevenue'] = df['totals.transactionRevenue'].fillna(0).astype(float)/1e6

for col in ['trafficSource.isTrueDirect', 'device.isMobile']:
    df[col] = df[col].fillna(False)
    
print(df['totals.transactionRevenue'].min())
print(df['totals.transactionRevenue'].max())

In [ ]:
# 检查异常值
print(df['totals.transactionRevenue'].min(), df['totals.transactionRevenue'].max())

# 检查检查是否存在取整后归零的订单
small = df[(df['totals.transactionRevenue'] > 0) & (df['totals.transactionRevenue'] < 1)]
print(len(small), small['totals.transactionRevenue'].tolist())

In [ ]:
# 替换缺失字符    
str_cols = df.select_dtypes(include=['object']).columns  
replace_list = ['(not set)', 'not available in demo dataset', '(none)']

for col in str_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(replace_list, 'Unknown')
    df[col] = df[col].replace(['nan', 'None'], 'Unknown')

ad_cols = ['trafficSource.adwordsClickInfo.slot', 'trafficSource.adwordsClickInfo.adNetworkType']
for col in ad_cols:
    df[col] = df[col].replace('Unknown', 'Non-Ad')

df['channelGrouping'] = df['channelGrouping'].replace('(Other)', 'Others')

In [ ]:
# 删除重复值
df = df.drop_duplicates()

##### 5.补充时间日期字段

In [ ]:
df['date'] = df['visitStartTime'].dt.date
df['Year'] = df['visitStartTime'].dt.year
df['Month'] = df['visitStartTime'].dt.month
df['Day'] = df['visitStartTime'].dt.day
df['Hour'] = df['visitStartTime'].dt.hour
df['Day_of_week_num'] = df['visitStartTime'].dt.dayofweek+1

dow_map = {1:'Mon', 2:'Tue', 3:'Wed', 4:'Thu', 5:'Fri', 6:'Sat', 7:'Sun'}
df['Day_of_week'] = df['Day_of_week_num'].map(dow_map)
df['is_weekday'] = df['Day_of_week_num'].apply(lambda x: 'Weekday' if x<6 else 'Weekend')

In [ ]:
df.info()

### 三、导出

基于最终制作看板的需求，剔除未参与分析的字段，完成二次收敛

In [ ]:
col = ['device.browser', 'device.isMobile', 'geoNetwork.region', 'trafficSource.campaign', 
       'trafficSource.isTrueDirect', 'trafficSource.adwordsClickInfo.slot', 
       'trafficSource.adwordsClickInfo.adNetworkType']
df.drop(columns=col, inplace=True)

In [ ]:
OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_DIR/'Cleaned_Google_Dataset.csv', index=False, encoding='utf-8')

### 四、创建客户维度表并导出

In [ ]:
df2 = df.groupby('fullVisitorId', as_index=False).agg(min_visit_number=('visitNumber', 'min')
                                      ,max_visit_number=('visitNumber', 'max')
                                      ,session_count_in_dataset=('fullVisitorId', 'size')
                                      ,total_transaction_revenue=('totals.transactionRevenue','sum'))
df2['customer_origin'] = df2['min_visit_number'].apply(lambda x: "New" if x==1 else "Pre-existing")
df2['is_purchaser'] = df2['total_transaction_revenue'].apply(lambda x: 1 if x> 0 else 0)

In [ ]:
df3 = df[df['totals.transactionRevenue']>0].sort_values(['fullVisitorId', 'visitNumber']).drop_duplicates(subset='fullVisitorId', keep='first')[[
    'fullVisitorId','visitNumber','totals.transactionRevenue']].rename(columns={'visitNumber':'first_purchase_visit_number',
                        'totals.transactionRevenue':'first_purchase_amount'})
df2 = df2.merge(df3, on='fullVisitorId', how='left')
df2['first_purchase_visit_number'] = df2['first_purchase_visit_number'].astype('Int64')
df2['fullVisitorId'] = df2['fullVisitorId'].astype(str)
df2

In [ ]:
df2.to_csv(OUTPUT_DIR/'Customers.csv', index=False, encoding='utf-8')